In [1]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import timm
from torchvision import transforms
from sklearn.metrics import roc_auc_score

/Users/masha/Documents/visual-reasoning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --- Config ---
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Running on {DEVICE}")

IMG_SIZE = 64
NUM_SHAPES = 4
N_SAMPLES = 200
BATCH_SIZE = 8
SEED = 42

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Colors dataset (from fm.ipynb) ---
def random_target_image_np(h: int, w: int, num_shapes: int = 4) -> np.ndarray:
    img = np.zeros((h, w, 3), dtype=np.float32)
    for _ in range(num_shapes):
        color = np.random.rand(3).astype(np.float32)
        y0 = random.randint(0, h - 8)
        x0 = random.randint(0, w - 8)
        y1 = min(h, y0 + random.randint(6, h // 2))
        x1 = min(w, x0 + random.randint(6, w // 2))
        img[y0:y1, x0:x1, :] = color
    return img


def rotate_np(img: np.ndarray, angle_deg: float) -> np.ndarray:
    h, w, _ = img.shape
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    rotated = cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0.0,
    )
    return rotated


def to_tensor(img: np.ndarray) -> torch.Tensor:
    # HWC -> CHW, then normalize to [-1, 1]
    t = torch.from_numpy(img).float().permute(2, 0, 1).contiguous()
    return (t - 0.5) / 0.5


class ColorsReasoningDataset(Dataset):
    def __init__(self, n_samples: int = 200, image_size: int = 64, num_shapes: int = 4):
        self.data = []
        for _ in range(n_samples):
            img_a_np = random_target_image_np(image_size, image_size, num_shapes)
            angle = random.randint(0, 359)
            img_b_np = rotate_np(img_a_np, angle)

            is_same = (random.random() > 0.5)
            if not is_same:
                img_b_np = cv2.flip(img_b_np, 1)  # mirror mismatch

            self.data.append((to_tensor(img_a_np), to_tensor(img_b_np), 1.0 if is_same else 0.0))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


# --- DINOv3 baseline ---
dino = timm.create_model("vit_small_patch16_dinov3", pretrained=True).to(DEVICE).eval()


def get_dino_embedding(img_64_tensor: torch.Tensor) -> torch.Tensor:
    # img_64_tensor: (B, 3, 64, 64) in [-1, 1]
    img = (img_64_tensor * 0.5) + 0.5  # -> [0, 1]
    img = F.interpolate(img, size=(224, 224), mode="bilinear", align_corners=False)
    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    img = norm(img)
    with torch.no_grad():
        emb = dino.forward_features(img)[:, 0, :]
    return emb


test_loader = DataLoader(ColorsReasoningDataset(N_SAMPLES, IMG_SIZE, NUM_SHAPES), batch_size=BATCH_SIZE, shuffle=False)

y_true = []
y_scores = []

with torch.no_grad():
    for img_a, img_b, label in test_loader:
        img_a = img_a.to(DEVICE)
        img_b = img_b.to(DEVICE)

        emb_a = get_dino_embedding(img_a)
        emb_b = get_dino_embedding(img_b)

        sims = F.cosine_similarity(emb_a, emb_b, dim=1)
        y_scores.extend(sims.cpu().numpy().tolist())
        y_true.extend(label.numpy().tolist())

auc = roc_auc_score(y_true, y_scores)

# Best threshold for accuracy
scores = np.array(y_scores)
labels = np.array(y_true)
thresholds = np.unique(scores)

best_acc = 0.0
best_thresh = thresholds[0] if len(thresholds) > 0 else 0.0
for t in thresholds:
    preds = (scores >= t).astype(np.float32)
    acc = (preds == labels).mean()
    if acc > best_acc:
        best_acc = acc
        best_thresh = float(t)

print(f"DINOv3 baseline AUC: {auc:.4f}")
print(f"Match/mismatch accuracy: {best_acc:.2%}")

Running on mps
DINOv3 baseline AUC: 0.4593
Match/mismatch accuracy: 52.00%
